<a href="https://colab.research.google.com/github/mybenkhadda/GEMS---drivendata/blob/main/notebooks/colab/GEMS_DISCOVERY_GEOMOE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GEMS discovery-first GeoMoE (code rev dgm.2-r1)

**One notebook, one CONFIG, run top to bottom.** `FAST_DIAGNOSTIC` measures candidate coverage first ([00]-[06]); `FULL` continues from the same cached artifacts into fusion, gating, deployment and export.

| | |
|---|---|
| Runtime | Colab **CPU**, High-RAM. No GPU, no neural network is trained. |
| Reference | **A_submitted_W2** = the frozen submitted W2 policy (public DTI 0.0942), one cell for every scenario |
| Outputs | `MyDrive/GEMS_2/outputs/discovery_geomoe/` |
| Code | `gems_bench/geomoe/` - docs `docs/geomoe/DISCOVERY_GEOMOE.md` |

Every stage checks its upstream stages: a failed / blocked dependency gives `BLOCKED_UPSTREAM` (compute) or `NOT_EVALUATED` (decision) and **no conclusion**. A failing test gives `TESTS_FAILED` and blocks candidates, modeling, selection and export. Cached results are reused only after validation (`REUSED_VALID_CACHE`).

**After a disconnect:** restart, run cells 1-2, then the stage cells again (finished units / stages are reused).

## 1 - Configuration (the only cell you edit)

`mode`: `FAST_DIAGNOSTIC` ([00]-[06] + integrity + reports) | `FULL` | `RESUME` (= FULL, cache forced). dgm.1 keys (`experts`, `experiments`, `export`) still work.

In [ ]:
CONFIG = {
    'repo': 'mybenkhadda/GEMS---drivendata',
    'ref': 'main',                          # pin the pushed commit SHA for a reproducible run
    'drive_root': '/content/drive/MyDrive/GEMS_2',
    'local_root': '/content/gems_work',

    'mode': 'FAST_DIAGNOSTIC',              # FAST_DIAGNOSTIC | FULL | RESUME
    'reuse_cached': True,
    'train_neural_models': False,           # True is refused

    'backfill_missing_g02': False,          # True: compute ONLY the missing G02 unit rasters (frozen G02 config)
    'dem': 'cached_only',                   # cached_only | off (never a DEM download)

    'run_synthetic_tests': True,            # self-tests + pytest, ENFORCED
    'strict_dependency_checks': True,

    'export_validated_candidate': True,
    'allow_exploratory_export': False,

    'save_all_outputs': True,
    'sync_to_drive': True,
    'overrides': {},                        # optional overrides of configs/geomoe/discovery_geomoe.yaml
}
PRIVATE_REPO = False
EXPECT = {'code_rev': 'dgm.2-r1', 'commit': None}   # set 'commit' to the pushed SHA

## 2 - Environment, Drive, frozen engine loader

In [ ]:
import base64, json, os, subprocess, sys
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
CODE = '/content/GEMS'
REPO_URL = 'https://github.com/' + CONFIG['repo'] + '.git'

def _git_env():
    if not PRIVATE_REPO:
        return None
    from google.colab import userdata
    hdr = base64.b64encode(('x-access-token:' + (userdata.get('GITHUB_TOKEN') or '')).encode()).decode()
    return {**os.environ, 'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
            'GIT_CONFIG_VALUE_0': 'AUTHORIZATION: basic ' + hdr}

def git(*args, auth=False, check=True):
    r = subprocess.run(['git', *args], capture_output=True, text=True, env=_git_env() if auth else None)
    if check and r.returncode:
        raise RuntimeError('git ' + ' '.join(args[:4]) + ' failed:\n' + r.stderr[-1500:])
    return r

if not Path(CODE, '.git').exists():
    git('clone', '-q', REPO_URL, CODE, auth=True)
git('-C', CODE, 'fetch', '-q', '--tags', 'origin', auth=True)
ref = CONFIG['ref']
if git('-C', CODE, 'rev-parse', '--verify', '-q', 'refs/remotes/origin/' + ref, check=False).returncode == 0:
    git('-C', CODE, 'checkout', '-q', '-B', 'geomoe_colab_active', 'origin/' + ref)
else:
    git('-C', CODE, 'checkout', '-q', '--detach', ref)
HEAD = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
if 'gems_bench' in sys.modules:
    raise RuntimeError('gems_bench already imported: Runtime -> Restart session, then run from the top')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', CODE + '[colab]', 'pytest'], check=True)
sys.path.insert(0, CODE)
os.environ['GEMS_CODE_COMMIT'] = HEAD

import pandas as pd
from IPython.display import Image, Markdown, display
import gems_bench.geomoe as GM
from gems_bench.geomoe import run as GR
from gems_bench.leaderboard_focus import factory as FA
assert GM.GEOMOE_CODE_REV == EXPECT['code_rev'], f'code rev {GM.GEOMOE_CODE_REV} != {EXPECT["code_rev"]}'
assert not EXPECT['commit'] or HEAD.startswith(EXPECT['commit']), f'HEAD {HEAD} != {EXPECT["commit"]}'
NB = GR.notebook_config(CONFIG)
F = FA.Factory(GR.factory_config(NB), HEAD)     # frozen-engine loader only (no factory candidate is exported)
G = GR.DiscoveryGeoMoE(CONFIG, HEAD, factory=F)
RUN_ALL = G.run                                   # optional one-call entry point == every cell below
OUT = G.ws.root
FULL = NB['mode'] != 'FAST_DIAGNOSTIC'

def show(rel, cols=None, n=40):
    p = OUT / rel
    if not p.exists():
        print('ABSENT:', rel); return None
    df = pd.read_csv(p) if p.suffix == '.csv' else pd.read_parquet(p)
    display(df[[c for c in (cols or df.columns) if c in df]].head(n)); return df

print('SHA', HEAD, '| code rev', GM.GEOMOE_CODE_REV, '| mode', NB['mode'])
print('Drive outputs:', G.drive / G.out)

## 3 - [00] Synthetic tests (enforced)

Runtime self-tests + `pytest tests/test_discovery_geomoe.py`. A nonzero pytest exit code or a failed critical self-test = `TESTS_FAILED`: the full output is saved to `audit/pytest_output.txt`, the failing tests are listed, and stages [04]-[12] are blocked.

In [ ]:
M00 = G.s00_tests()
if G.status['00_tests'] == 'TESTS_FAILED':
    print('\nTESTS_FAILED - modeling, selection and export are blocked. Failing:')
    for w in M00.get('failing', []): print('  ', w)
    print((OUT / 'audit/pytest_output.txt').read_text()[-4000:])

## 4 - [01]-[03] Frozen artifacts, audit (+ optional G02 backfill), scenarios

In [ ]:
G.s01_attach(); G.s02_audit(); G.s03_scenarios()
show('audit/component_inventory.csv')
print(json.dumps(G.ws.read_json('audit/g02_backfill.json'), indent=1, default=str))
show('validation/scenarios.csv')

## 5 - [04] Candidate generation (validated unit cache)

Per unit: label-free layers -> pools -> inventory (saved first) -> coverage parts (saved second). A unit is reused only when its fingerprint, arrays and tables validate; only invalid parts are recomputed.

In [ ]:
G.s04_candidates()
show('candidates/source_provenance.csv', n=60)

## 6 - [05] Oracle analysis + frozen submitted W2 + [06] bottleneck diagnosis

In [ ]:
G.s05_oracle(); G.s06_decision()
show('validation/frozen_w2_baseline.csv', ['scenario', 'zone', 'rep', 'frac', 'area_frac_eligible', 'dti', 'policy_source'])
show('audit/baseline_policy_parity.csv')
show('candidates/oracle_coverage.csv')
show('audit/candidate_bottleneck_reconciliation.csv')
show('validation/v2_coverage_by_subtype.csv')
print(json.dumps(G.ws.read_json('candidates/coverage_decision.json'), indent=1, default=str)[:3000])

## 7 - [07]-[10] Experts, static fusion, geological rules, learned GeoMoE (FULL only)

In [ ]:
if FULL:
    G.s07_experts(); G.s08_static(); G.s09_rules(); G.s10_learned()
    A = show('experts/source_availability.csv', n=0)
    if A is not None: display(A[A.member == '*'])
else:
    print('FAST_DIAGNOSTIC: stages [07]-[12] not run (set mode FULL to continue from the cached artifacts)')

## 8 - [10b] Model comparison and selection, [11] deployment, [12] submission (FULL only)

Fixed-policy evaluation (one rule for every scenario, development units only) vs the frozen submitted W2; paired block bootstrap; effect decomposition; V2 by subtype. Export only of the selected model (or EXPLORATORY when allowed); nothing is uploaded.

In [ ]:
if FULL:
    G.s10b_compare(); G.s11_deployment(); G.s12_export()
    show('gating/model_summary.csv', ['scenario', 'model', 'protocol', 'dti', 'worst_zone_dti', 'params_all'])
    show('gating/paired_comparison.csv'); show('gating/effect_decomposition.csv'); show('validation/v2_by_subtype.csv')
    print(json.dumps({k: G.ws.read_json('gating/selection.json').get(k) for k in ('selected', 'status', 'measured', 'reasons')}, indent=1, default=str))
    show('submissions/submission_manifest.csv')

## 9 - [14] Stage integrity, [13] reports and figures

In [ ]:
G.s14_integrity(); G.s13_reports()
show('audit/stage_integrity.csv', ['stage', 'actual_status', 'missing_outputs', 'failure_reason', 'affected_file', 'scientific_conclusion_allowed'])
for f in ('baseline_comparison', 'oracle_vs_actual', 'unique_geology_recovery', 'v2_fault_subtypes', 'gating_comparison', 'geographic_robustness'):
    p = OUT / 'figures' / f'{f}.png'
    if p.exists(): display(Image(str(p)))
for r in ('EXECUTION_STATUS', 'SCIENTIFIC_FINDINGS', 'LEADERBOARD_RECOMMENDATION', 'NEXT_ACTION'):
    display(Markdown((OUT / f'reports/{r}.md').read_text()))
G.final_message()